In [ ]:
######chuli data

In [ ]:
import numpy as np

data = np.load("/home/mengyao.li/project/compression/data/v-wideband/case_CDL-30_frame99 (copy 1).npy")  # 假设数据文件名为"your_data.npy"

print("原始数据形状:", data.shape)  

# 提取实部和虚部
real_parts = data.real  # 实部，形状为(1000, 1, 4)
imag_parts = data.imag  # 虚部，形状为(1000, 1, 4)

#将实部和虚部按最后一个维度拼接，得到8个数值
extracted_data = np.concatenate([real_parts, imag_parts], axis=-1)

# 可选：如果需要去除中间的维度(1)，可以使用squeeze
# extracted_data = extracted_data.squeeze(axis=1)  # 形状变为(1000, 8)

# 验证结果形状
print("提取实部虚部后的数据形状:", extracted_data.shape) 

# 保存处理后的数据（可选）
np.save("extracted_data.npy", extracted_data)





In [ ]:
import numpy as np
import os

# 1. 定义路径
data_dir = "/home/mengyao.li/project/compression/data/v-wideband"
output_file = "/home/mengyao.li/project/compression/data/processed_v_wideband.npy"

# 2. 收集所有.npy文件路径
npy_files = [f for f in os.listdir(data_dir) if f.endswith(".npy")]
npy_files = [os.path.join(data_dir, f) for f in npy_files]

# 3. 输出文件数量
#print(f"文件夹下共有 {len(npy_files)} 个.npy数据文件")

# 4. 批量处理每个文件：提取实部虚部
all_data = []
for file_path in npy_files:
    # 加载数据
    data = np.load(file_path)
    
    # 提取实部和虚部
    real_parts = data.real  
    imag_parts = data.imag  
    
    # 拼接实部和虚部
    processed = np.concatenate([real_parts, imag_parts], axis=-1) 
    
    # 添加到列表
    all_data.append(processed)

# 5. 合并所有数据
if not all_data:
    print("错误：没有找到有效数据文件")
else:
    # 合并为一个数组
    final_data = np.concatenate(all_data, axis=0)
    
    # 6. 输出整理后的形状
    #print(f"整理后的数据形状：{final_data.shape}")
    
    # 7. 保存结果
    np.save(output_file, final_data)
    #print(f"数据已保存到：{output_file}")

 

In [ ]:
import numpy as np

# 加载数据
data = np.load("/home/mengyao.li/project/compression/data/processed_v_wideband.npy")
print(f"原始数据形状: {data.shape}")

# 查看是否有重复样本（将每个样本展平为一维，再判断重复）
data_flat = data.reshape(data.shape[0], -1)  # 展平为 (n_samples, 8)
_, unique_indices = np.unique(data_flat, axis=0, return_index=True)  # 找到唯一样本的索引
unique_data = data[np.sort(unique_indices)]  # 按原始顺序保留唯一样本

print(f"去重后数据形状: {unique_data.shape}")
print(f"删除的重复样本数: {data.shape[0] - unique_data.shape[0]}")

In [ ]:
######1124 real data
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import os

# 设置随机种子
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# 定义量化函数
class QuantizeSTEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bitwidth, quant_min, quant_max):
        levels = 2 ** bitwidth
        scale = (quant_max - quant_min) / (levels - 1)
        x_normalized = (x - quant_min) / scale
        ctx.save_for_backward(x_normalized)
        ctx.levels = levels
        x_quantized = torch.round(torch.clamp(x_normalized, 0, levels - 1))
        return x_quantized.long().squeeze()

    @staticmethod
    def backward(ctx, grad_output):
        x_normalized, = ctx.saved_tensors
        levels = ctx.levels
        # 创建mask：只在有效量化范围内传递梯度
        mask = (x_normalized >= 0) & (x_normalized <= levels - 1)
        grad_input = grad_output * mask.float()  # 应用mask到梯度
        return grad_input, None, None, None

class QuantizeSTELayer(nn.Module):
    def __init__(self, bitwidth, quant_min, quant_max):
        super(QuantizeSTELayer, self).__init__()
        self.bitwidth = bitwidth
        self.quant_min = quant_min
        self.quant_max = quant_max

    def forward(self, x):
        return QuantizeSTEFunction.apply(x, self.bitwidth, self.quant_min, self.quant_max)

# 调整模型以匹配输入形状 (batch_size, 1, 8)
class CodecSystem(nn.Module):
    def __init__(self):
        super(CodecSystem, self).__init__()
        
        # 编码器：
        self.encoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.Tanh(),
            nn.Linear(16, 8),
            nn.Tanh(),
            nn.Linear(8, 1),
        )
        
        # 解码器：
        self.decoder = nn.Sequential(
            nn.Linear(1, 8),
            nn.Tanh(),
            nn.Linear(8, 16),
            nn.Tanh(),
            nn.Linear(16, 8),
        )
        
        # 4比特量化配置
        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth
        self.quant_min = -2
        self.quant_max = 2
        self.quantizer = QuantizeSTELayer(self.bitwidth, self.quant_min, self.quant_max)

        self.output_scale = nn.Parameter(torch.tensor(1.0))
        self.output_bias = nn.Parameter(torch.tensor(0.0))

    def quantized_to_binary(self, quantized):
        if quantized.dim() == 0:
            quantized = quantized.unsqueeze(0)
        binary = torch.zeros(quantized.numel(), self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):
            binary[:, self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary.squeeze(0)

    def binary_to_quantized(self, binary):
        if binary.dim() == 1:
            binary = binary.unsqueeze(0)
        quantized = torch.zeros(binary.shape[0], dtype=torch.long, device=binary.device)
        for i in range(self.bitwidth):
            quantized = quantized * 2 + binary[:, i]
        return quantized.squeeze(0)

    def dequantize(self, x_quantized):
        if x_quantized.dim() == 0:
            x_quantized = x_quantized.unsqueeze(0)
        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

    def forward(self, x):
        # 输入形状: (batch_size, 1, 8)
        batch_size = x.shape[0]
        
        # 编码：(batch_size, 1, 8) -> (batch_size, 1)
        x_encoded = self.encoder(x.squeeze(1))  
        x_encoded = x_encoded * self.output_scale + self.output_bias
        
        # 量化
        x_quantized = self.quantizer(x_encoded)
        x_binary = self.quantized_to_binary(x_quantized)
        x_quantized_recovered = self.binary_to_quantized(x_binary)
        
        # 解量化
        x_dequantized = self.dequantize(x_quantized_recovered)
        x_dequantized = x_dequantized.unsqueeze(1)  
        
        # 解码：(batch_size, 1) -> (batch_size, 1, 8)
        x_decoded = self.decoder(x_dequantized)
        x_output = x_decoded.unsqueeze(1) 
        
        return x_output, x_quantized, x_binary

# 定义数据集类
class AudioDataset(Dataset):
    def __init__(self, data):
        self.data = torch.from_numpy(data).float()
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

# 主函数
if __name__ == "__main__":
    # 数据路径
    data_path = "/home/mengyao.li/project/compression/data/processed_v_wideband.npy"
    
    
    data = np.load(data_path)
    
    
    # 划分训练集和测试集 (80% 训练, 20% 测试)
    dataset = AudioDataset(data)
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    
    
    # 创建DataLoader
    batch_size = 128
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    
    # 创建模型
    codec = CodecSystem()
    
    # 检查是否有可用的GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    codec.to(device)
    #print(f"使用设备: {device}")
    
    # 损失函数和优化器
    criterion = nn.L1Loss()
    optimizer = optim.SGD(codec.parameters(), lr=0.005, momentum = 0.9)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=0.0005)
    
    # 训练循环
    num_epochs = 100
    codec.train()
    
    print("\n开始训练...")
    for epoch in range(num_epochs):
        running_loss = 0.0
        
        for i, batch_data in enumerate(train_loader):
            # 将数据移到设备上
            batch_data = batch_data.to(device)
            
            optimizer.zero_grad()
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            loss.backward()
            
            # 梯度裁剪
            torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=0.5)
            
            optimizer.step()
            
            running_loss += loss.item()
            
            # 每100个批次
            if (i + 1) % 100 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{i+1}/{len(train_loader)}], Loss: {loss.item():.6f}')
        
        # 更新学习率
        scheduler.step()
        
        # 计算 epoch 的平均损失
        epoch_loss = running_loss / len(train_loader)
        current_lr = scheduler.get_last_lr()[0]
        print(f'Epoch [{epoch+1}/{num_epochs}], Average Loss: {epoch_loss:.6f}, LR: {current_lr:.6f}')
    
    # 测试阶段
    print("\n开始测试...")
    codec.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for batch_data in test_loader:
            batch_data = batch_data.to(device)
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            total_loss += loss.item() * batch_data.size(0)
    
    # 计算平均测试损失
    average_test_loss = total_loss / len(test_dataset)
    print(f"测试集平均MSE: {average_test_loss:.6f}")
    

